# bimp · fixture detector (windows, doors, toilets, sinks, showers and bathtubs)

Trains a small object detector on **CubiCasa5K** and scores it on the Junction challenge's count questions.

- **Training data** comes straight from Zenodo (the original CubiCasa5K with its hand-drawn annotations), not from R2.
  The challenge's `cubicasa5k/*.png` files are the same plans renamed (`colorful-10052.png` = `colorful/10052`).
- **Fair score:** the model trains on CubiCasa's official *train* split; thresholds are tuned on *val*; the score is
  reported on *test* only, plans the model never saw.
- The only thing read from R2 is `dataset/questions.jsonl` (26 MB), to score against.
- Labels were checked: counting the annotations gives the challenge's answer on 546 of 546 sampled questions.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free tier is enough).
2. In Cloudflare: *R2 → Manage API tokens → Create API token*: permission **Object Read & Write**, bucket **bimp-files**.
   Copy the Access Key ID, Secret Access Key and the S3 endpoint (`https://<account id>.r2.cloudflarestorage.com`).
3. In Colab, the 🔑 **Secrets** panel (left bar): add `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT`, and switch on
   *Notebook access* for each. (Skip this and the notebook asks you to upload `questions.jsonl` instead.)
4. *Runtime → Run all*. Download ~5 min, conversion ~5 min, training ~1.5–2.5 h on a T4. Checkpoints go to your
   Google Drive, so a disconnect loses nothing: re-run all and training resumes.

Licence: CubiCasa5K is CC BY-NC-SA 4.0. The trained model inherits non-commercial use.

In [ ]:
!nvidia-smi -L || echo 'No GPU: Runtime → Change runtime type → T4 GPU'
!pip -q install "ultralytics>=8.3,<9" boto3 onnx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/bimp-ml'          # checkpoints, report and model survive disconnects here
os.makedirs(WORK, exist_ok=True)
%cd /content

## 1 · CubiCasa5K from Zenodo (5.4 GB)

In [ ]:
import os
if not os.path.exists('/content/cubicasa5k/train.txt'):
    !wget -q --show-progress -c -O /content/cubicasa5k.zip "https://zenodo.org/api/records/2613548/files/cubicasa5k.zip/content"
    !unzip -q -o /content/cubicasa5k.zip -d /content && rm /content/cubicasa5k.zip
!wc -l /content/cubicasa5k/{train,val,test}.txt

## 2 · Scripts (same as `ml/` in the repo)

In [ ]:
%%writefile cc_labels.py
"""CubiCasa5K model.svg -> object boxes (pixel coords of F1_original.png) for windows, doors and fittings."""
import re, xml.etree.ElementTree as ET

NUM = r'[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?'
def parse_transform(s):
    m = [1, 0, 0, 1, 0, 0]
    for name, args in re.findall(r'(\w+)\s*\(([^)]*)\)', s or ''):
        a = [float(v) for v in re.findall(NUM, args)]
        if name == 'matrix' and len(a) == 6: t = a
        elif name == 'translate': t = [1, 0, 0, 1, a[0], a[1] if len(a) > 1 else 0]
        elif name == 'scale': t = [a[0], 0, 0, a[1] if len(a) > 1 else a[0], 0, 0]
        elif name == 'rotate':
            import math
            r = math.radians(a[0]); c, s_ = math.cos(r), math.sin(r)
            t = [c, s_, -s_, c, 0, 0]
            if len(a) == 3:
                cx, cy = a[1], a[2]
                t = mul(mul([1, 0, 0, 1, cx, cy], t), [1, 0, 0, 1, -cx, -cy])
        else: continue
        m = mul(m, t)
    return m
def mul(p, q):
    a, b, c, d, e, f = p; A, B, C, D, E, F = q
    return [a*A + c*B, b*A + d*B, a*C + c*D, b*C + d*D, a*E + c*F + e, b*E + d*F + f]
def apply(m, x, y): return m[0]*x + m[2]*y + m[4], m[1]*x + m[3]*y + m[5]

def points_of(el):
    tag = el.tag.split('}')[-1]
    if tag in ('polygon', 'polyline'):
        v = [float(x) for x in re.findall(NUM, el.get('points', ''))]; return list(zip(v[::2], v[1::2]))
    if tag == 'rect':
        x, y, w, h = (float(el.get(k, 0)) for k in ('x', 'y', 'width', 'height')); return [(x, y), (x+w, y+h)]
    if tag == 'circle':
        cx, cy, r = (float(el.get(k, 0)) for k in ('cx', 'cy', 'r')); return [(cx-r, cy-r), (cx+r, cy+r)]
    if tag == 'line':
        return [(float(el.get('x1', 0)), float(el.get('y1', 0))), (float(el.get('x2', 0)), float(el.get('y2', 0)))]
    return []

def objects(svg_text):
    """[(class, (x0, y0, x1, y1))] for every element whose class starts with Window, Door or FixedFurniture."""
    root = ET.fromstring(svg_text)
    out = []
    def walk(el, m):
        m = mul(m, parse_transform(el.get('transform')))
        cls = (el.get('class') or '').strip()
        if re.match(r'(Window|Door|FixedFurniture)\b', cls) and not cls.startswith('FixedFurnitureSet'):
            # outline: the BoundaryPolygon if present (furniture), else the first polygon (windows, doors)
            bp = None
            for ch in el.iter():
                if (ch.get('class') or '') == 'BoundaryPolygon': bp = ch; break
            target = bp if bp is not None else el
            pts = []
            def collect(e, mm):
                mm = mul(mm, parse_transform(e.get('transform'))) if e is not target else mm
                pts.extend(apply(mm, x, y) for x, y in points_of(e))
                for c in e: collect(c, mm)
            if bp is not None:
                # the boundary's own ancestors between el and bp may carry transforms; they are rare, apply bp's
                collect(bp, mul(m, parse_transform(bp.get('transform'))))
            else:
                first = next((c for c in el if c.tag.split('}')[-1] in ('polygon', 'rect')), None)
                if first is not None: pts = [apply(m, x, y) for x, y in points_of(first)]
                else:  # no outline of its own: the bounds of everything it draws
                    def every(e, mm):
                        for c in e:
                            cm = mul(mm, parse_transform(c.get('transform')))
                            pts.extend(apply(cm, x, y) for x, y in points_of(c)); every(c, cm)
                    every(el, m)
            if pts:
                xs, ys = [p[0] for p in pts], [p[1] for p in pts]
                out.append((cls, (min(xs), min(ys), max(xs), max(ys))))
            if not cls.startswith('FixedFurniture'): return  # door/window parts are not objects; furniture can contain furniture (a sink in a cabinet)
        for ch in el: walk(ch, m)
    walk(root, [1, 0, 0, 1, 0, 0])
    return out

In [ ]:
%%writefile classes.py
import re
# Detector classes and how CubiCasa5K annotation classes map to them (checked against the challenge's answers).
NAMES = ['window', 'door', 'toilet', 'sink', 'shower_bath']
def cat(c):
    if c.startswith('Window'): return 'windows'
    if c.startswith('Door') and not c.startswith('Door None'): return 'doors'  # "Door None" = open doorway, no leaf
    if c == 'FixedFurniture Toilet': return 'toilets'
    if re.match(r'FixedFurniture \w*Sink(Left|Right)?$', c): return 'sinks'  # Sink, DoubleSink, RoundSink, CornerSink, SideSink…; a double bowl is one
    if re.match(r'FixedFurniture (Shower|Bathtub|BathtubRound)$', c): return 'wet'  # ShowerCab is the enclosure of a Shower
    return None
CLASS_ID = {'windows': 0, 'doors': 1, 'toilets': 2, 'sinks': 3, 'wet': 4}

In [ ]:
%%writefile prepare.py
"""CubiCasa5K (Zenodo 2613548) -> YOLO dataset of windows, doors, toilets, sinks, showers/bathtubs.

Uses the official train/val/test split. Images: F1_scaled.png (model.svg coordinates are in its pixel space).
  python prepare.py /path/to/cubicasa5k /path/to/out
"""
import os, sys, shutil, collections
from concurrent.futures import ProcessPoolExecutor
from PIL import Image
from cc_labels import objects
from classes import CLASS_ID, NAMES, cat

Image.MAX_IMAGE_PIXELS = None

def convert(args):
    src, out, split, plan = args
    d = os.path.join(src, plan.strip('/'))
    img, svg = os.path.join(d, 'F1_scaled.png'), os.path.join(d, 'model.svg')
    if not (os.path.exists(img) and os.path.exists(svg)): return plan, None
    W, H = Image.open(img).size
    rows = []
    for cls, (x0, y0, x1, y1) in objects(open(svg, encoding='utf-8', errors='replace').read()):
        k = cat(cls)
        if k is None: continue
        # thin openings: give boxes at least 6 px so they stay learnable
        cx, cy, w, h = (x0 + x1) / 2, (y0 + y1) / 2, max(x1 - x0, 6), max(y1 - y0, 6)
        x0, x1, y0, y1 = max(0, cx - w / 2), min(W, cx + w / 2), max(0, cy - h / 2), min(H, cy + h / 2)
        if x1 - x0 < 2 or y1 - y0 < 2: continue
        rows.append(f"{CLASS_ID[k]} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
    name = plan.strip('/').replace('/', '-')  # same names as the challenge files, e.g. colorful-10052
    os.makedirs(f'{out}/images/{split}', exist_ok=True); os.makedirs(f'{out}/labels/{split}', exist_ok=True)
    dst = f'{out}/images/{split}/{name}.png'
    if not os.path.exists(dst):
        try: os.link(img, dst)
        except OSError: shutil.copy(img, dst)
    open(f'{out}/labels/{split}/{name}.txt', 'w').write('\n'.join(rows))
    return plan, len(rows)

if __name__ == '__main__':
    src, out = sys.argv[1], sys.argv[2]
    limit = int(os.environ.get('LIMIT', '0'))
    jobs = []
    for split in ('train', 'val', 'test'):
        plans = [l.strip() for l in open(os.path.join(src, f'{split}.txt')) if l.strip()]
        jobs += [(src, out, split, p) for p in (plans[:limit] if limit else plans)]
    stats = collections.Counter()
    with ProcessPoolExecutor() as ex:
        for plan, n in ex.map(convert, jobs, chunksize=16):
            stats['missing' if n is None else 'ok'] += 1
    open(f'{out}/data.yaml', 'w').write(f"path: {os.path.abspath(out)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames: {NAMES}\n")
    print(dict(stats), '->', out)

In [ ]:
%%writefile evaluate.py
"""Score a trained detector on the challenge's count questions, fairly.

Thresholds are tuned on CubiCasa's val split; the score is reported on its test split only (plans the model never
trained on). Images: F1_original.png (the files in the challenge dataset are these, renamed).
  python evaluate.py best.pt /path/to/cubicasa5k questions.jsonl [out.json]
"""
import json, re, sys, collections
import numpy as np
from ultralytics import YOLO
from classes import NAMES

QUESTIONS = {  # English wording; the Finnish twin of each question has the same answer
    'window': r'^How many windows', 'door': r'^How many doors', 'toilet': r'toilet bowls',
    'sink': r'sinks and wash basins', 'shower_bath': r'showers and bathtubs',
}

def answers(path):
    out = collections.defaultdict(dict)
    for line in open(path):
        r = json.loads(line)
        if '/cubicasa5k/' not in r['path']: continue
        name = r['path'].rsplit('/', 1)[-1][:-4]
        for k, pat in QUESTIONS.items():
            if re.search(pat, r['question']): out[name][k] = int(r['answer'])
    return out

def confidences(model, src, split, imgsz):
    """Per plan: for each class, the sorted confidences of its detections (so any threshold can be applied later)."""
    plans = [l.strip().strip('/') for l in open(f'{src}/{split}.txt') if l.strip()]
    res = {}
    for i in range(0, len(plans), 16):
        batch = plans[i:i + 16]
        preds = model.predict([f'{src}/{p}/F1_original.png' for p in batch], imgsz=imgsz, conf=0.05, iou=0.5, max_det=600, verbose=False)
        for p, pr in zip(batch, preds):
            c = collections.defaultdict(list)
            for cls, conf in zip(pr.boxes.cls.tolist(), pr.boxes.conf.tolist()): c[NAMES[int(cls)]].append(conf)
            res[p.replace('/', '-')] = c
    return res

def accuracy(conf, ans, k, t):
    hits = [len([c for c in conf[n][k] if c >= t]) == a[k] for n, a in ans.items() if n in conf and k in a]
    return (sum(hits) / len(hits), len(hits)) if hits else (None, 0)

if __name__ == '__main__':
    weights, src, qpath = sys.argv[1:4]
    imgsz = 1024
    model, ans = YOLO(weights), answers(qpath)
    val, test = confidences(model, src, 'val', imgsz), confidences(model, src, 'test', imgsz)
    grid = np.round(np.arange(0.1, 0.9, 0.025), 3)
    report = {}
    for k in QUESTIONS:
        best = max(grid, key=lambda t: accuracy(val, ans, k, t)[0] or 0)
        acc, n = accuracy(test, ans, k, best)
        report[k] = {'threshold': float(best), 'val_exact': accuracy(val, ans, k, best)[0], 'test_exact': acc, 'test_questions': n}
        print(f"{k:12} threshold {best:.3f}  val {report[k]['val_exact']:.3f}  test {acc:.3f} on {n} plans")
    if len(sys.argv) > 4: json.dump(report, open(sys.argv[4], 'w'), indent=2)

## 3 · Annotations → YOLO labels

In [ ]:
!python prepare.py /content/cubicasa5k /content/yolo
!ls /content/yolo/images/train | wc -l; head -3 /content/yolo/labels/train/$(ls /content/yolo/labels/train | head -1)

## 4 · Challenge questions (from R2, or upload the file)

In [ ]:
import os
Q = '/content/questions.jsonl'
try:
    from google.colab import userdata
    import boto3
    s3 = boto3.client('s3', endpoint_url=userdata.get('R2_ENDPOINT'), aws_access_key_id=userdata.get('R2_ACCESS_KEY_ID'),
                      aws_secret_access_key=userdata.get('R2_SECRET_ACCESS_KEY'), region_name='auto')
    s3.download_file('bimp-files', 'dataset/questions.jsonl', Q)
except Exception as e:
    print('R2 not available (', type(e).__name__, '): upload questions.jsonl instead')
    s3 = None
    from google.colab import files
    up = files.upload(); os.rename(next(iter(up)), Q)
!wc -l {Q}

## 5 · Train (resumes automatically after a disconnect)

In [ ]:
from ultralytics import YOLO
last = f'{WORK}/runs/fixtures/weights/last.pt'
if os.path.exists(last):
    model = YOLO(last); model.train(resume=True)
else:
    model = YOLO('yolo11s.pt')
    model.train(data='/content/yolo/data.yaml', imgsz=1024, epochs=80, patience=20, batch=-1, cos_lr=True,
                mosaic=0.5, degrees=0, fliplr=0.5, flipud=0.5, hsv_h=0, hsv_s=0, hsv_v=0.2,  # line drawings: flips fine, colour irrelevant
                project=f'{WORK}/runs', name='fixtures', exist_ok=True, plots=True)
best = f'{WORK}/runs/fixtures/weights/best.pt'

## 6 · Score on the challenge's count questions (test split only)

In [ ]:
!python evaluate.py {best} /content/cubicasa5k {Q} {WORK}/report.json

## 7 · Export for the ingest container and upload to R2

In [ ]:
onnx = YOLO(best).export(format='onnx', imgsz=1024, dynamic=False, simplify=True)
import shutil; shutil.copy(onnx, f'{WORK}/fixtures-v1.onnx'); shutil.copy(f'{WORK}/report.json', '/content/report.json')
if s3:
    s3.upload_file(onnx, 'bimp-files', 'models/fixtures-v1.onnx')
    s3.upload_file(f'{WORK}/report.json', 'bimp-files', 'models/fixtures-v1.report.json')
    print('Uploaded to R2: models/fixtures-v1.onnx and models/fixtures-v1.report.json')
else:
    print('Saved to Google Drive:', WORK)